# LMSYS Chatbot Arena - LLM Preference Prediction

In [2]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/kaggle/input/llm-classification-finetuning")   # adjust if needed
OUTPUT_DIR = Path("/kaggle/working")
TRAIN_CSV  = INPUT_DIR / "train.csv"
TEST_CSV   = INPUT_DIR / "test.csv"
# ─────────────────────────────────────────────────────────────────────────────

print("Paths configured.")
print(f"  Train exists: {TRAIN_CSV.exists()}")
print(f"  Test  exists: {TEST_CSV.exists()}")

Paths configured.
  Train exists: True
  Test  exists: True


## 1. Feature Engineering

In [3]:
def parse_json_field(value):
    """Safely parse JSON-encoded list fields."""
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return " ".join(str(x) for x in parsed)
        return str(parsed)
    except Exception:
        return str(value)


def extract_features(text: str) -> dict:
    """Extract quality-signal features from a single response text."""
    if not isinstance(text, str):
        text = ""

    # Length features
    char_len    = len(text)
    word_count  = len(text.split())
    sent_count  = len(re.findall(r'[.!?]+', text)) or 1

    # Structure / formatting signals
    has_code       = int(bool(re.search(r'```', text)))
    bullet_count   = len(re.findall(r'^\s*[-*•]\s', text, re.MULTILINE))
    numbered_steps = len(re.findall(r'^\s*\d+[.)]\s', text, re.MULTILINE))
    header_count   = len(re.findall(r'^#{1,4}\s', text, re.MULTILINE))

    # Quality signals
    avg_word_len   = np.mean([len(w) for w in text.split()]) if text.split() else 0
    avg_sent_len   = word_count / sent_count
    has_examples   = int(bool(re.search(r'\bfor example\b|\bfor instance\b|\be\.g\.\b', text, re.I)))
    has_sources    = int(bool(re.search(r'\baccording to\b|\bsource\b|\breference\b', text, re.I)))
    starts_direct  = int(not bool(re.match(r'^(sure|certainly|of course|absolutely|great)', text.strip(), re.I)))
    hedge_count    = len(re.findall(r'\bmight\b|\bcould\b|\bperhaps\b|\bmaybe\b', text, re.I))

    return {
        "char_len":       char_len,
        "word_count":     word_count,
        "has_code":       has_code,
        "bullet_count":   bullet_count,
        "numbered_steps": numbered_steps,
        "header_count":   header_count,
        "avg_word_len":   avg_word_len,
        "avg_sent_len":   avg_sent_len,
        "has_examples":   has_examples,
        "has_sources":    has_sources,
        "starts_direct":  starts_direct,
        "hedge_count":    hedge_count,
    }


def build_feature_df(df: pd.DataFrame) -> pd.DataFrame:
    """Build a feature DataFrame from raw competition data."""
    rows = []
    for _, row in df.iterrows():
        resp_a = parse_json_field(row["response_a"])
        resp_b = parse_json_field(row["response_b"])

        fa = extract_features(resp_a)
        fb = extract_features(resp_b)

        diff  = {f"diff_{k}": fa[k] - fb[k]             for k in fa}
        ratio = {f"ratio_{k}": (fa[k] + 1) / (fb[k] + 1) for k in fa}
        feat_a = {f"a_{k}": v for k, v in fa.items()}
        feat_b = {f"b_{k}": v for k, v in fb.items()}

        rows.append({"id": row["id"], **feat_a, **feat_b, **diff, **ratio})

    return pd.DataFrame(rows)

print("Feature functions defined.")

Feature functions defined.


## 2. Heuristic Scorer
Works without any training data. Uses response quality signals to assign preference scores.

In [4]:
def heuristic_score(row: pd.Series) -> tuple:
    """
    Returns (score_a, score_b) — higher means that model is preferred.
    """
    score_a, score_b = 0.0, 0.0

    # Length sweet-spot: prefer ~200-800 words; penalise very short/long
    def length_score(wc):
        if wc < 20:   return -2.0
        if wc < 50:   return -0.5
        if wc < 800:  return  1.0
        if wc < 1500: return  0.5
        return -0.5

    score_a += length_score(row["a_word_count"])
    score_b += length_score(row["b_word_count"])

    # Structured responses are preferred
    score_a += 0.5 * row["a_has_code"]
    score_b += 0.5 * row["b_has_code"]
    score_a += 0.3 * min(row["a_numbered_steps"], 5)
    score_b += 0.3 * min(row["b_numbered_steps"], 5)
    score_a += 0.2 * min(row["a_bullet_count"], 5)
    score_b += 0.2 * min(row["b_bullet_count"], 5)
    score_a += 0.2 * min(row["a_header_count"], 3)
    score_b += 0.2 * min(row["b_header_count"], 3)

    # Direct, confident answers preferred
    score_a += 0.3 * row["a_starts_direct"]
    score_b += 0.3 * row["b_starts_direct"]
    score_a -= 0.1 * min(row["a_hedge_count"], 5)
    score_b -= 0.1 * min(row["b_hedge_count"], 5)

    # Examples add value
    score_a += 0.3 * row["a_has_examples"]
    score_b += 0.3 * row["b_has_examples"]

    return score_a, score_b


def scores_to_probs(score_a: float, score_b: float, tie_weight: float = 0.22) -> tuple:
    """
    Convert raw heuristic scores to (p_a, p_b, p_tie) via softmax.
    """
    exp_a = np.exp(score_a)
    exp_b = np.exp(score_b)
    total = exp_a + exp_b

    raw_a = exp_a / total
    raw_b = exp_b / total

    p_a   = raw_a * (1 - tie_weight)
    p_b   = raw_b * (1 - tie_weight)
    p_tie = tie_weight

    s = p_a + p_b + p_tie
    return p_a / s, p_b / s, p_tie / s

print("Heuristic scorer defined.")

Heuristic scorer defined.


## 3. (Optional) XGBoost ML Model

In [5]:
def train_xgb(feat_df: pd.DataFrame, labels: pd.Series):
    """Train XGBoost on engineered features. Returns (model, feature_cols)."""
    try:
        from xgboost import XGBClassifier
    except ImportError:
        print("XGBoost not available; skipping ML model.")
        return None, None

    feature_cols = [c for c in feat_df.columns if c != "id"]
    X = feat_df[feature_cols].fillna(0)
    y = labels  # 0=model_a, 1=model_b, 2=tie

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X, y)
    print(f"XGBoost trained on {len(X)} samples with {len(feature_cols)} features.")
    return model, feature_cols

print("XGBoost trainer defined.")

XGBoost trainer defined.


## 4. Main Pipeline — Load Data & Predict

In [6]:
print("Loading test data...")
test_df = pd.read_csv(TEST_CSV)
print(f"  Test samples: {len(test_df)}")

print("Building test features...")
test_feat = build_feature_df(test_df)
print(test_feat.shape)

Loading test data...
  Test samples: 3
Building test features...
(3, 49)


In [7]:
model = None
feature_cols = None

if TRAIN_CSV.exists():
    print("Loading train data...")
    train_df = pd.read_csv(TRAIN_CSV)
    print(f"  Train samples: {len(train_df)}")

    print("Columns:", train_df.columns.tolist())

    # Detect label column — handle both formats:
    # Format A: single "winner" column with values model_a / model_b / tie
    # Format B: separate winner_model_a, winner_model_b, winner_tie columns (0/1)
    if "winner" in train_df.columns:
        label_map = {"model_a": 0, "model_b": 1, "tie": 2}
        train_df["label"] = train_df["winner"].map(label_map)
    elif all(c in train_df.columns for c in ["winner_model_a", "winner_model_b", "winner_tie"]):
        # Convert one-hot style columns to single label
        def row_to_label(r):
            if r["winner_model_a"] == 1: return 0
            if r["winner_model_b"] == 1: return 1
            if r["winner_tie"]     == 1: return 2
            return -1  # unknown
        train_df["label"] = train_df.apply(row_to_label, axis=1)
    else:
        raise ValueError(f"Cannot find winner column(s). Columns: {train_df.columns.tolist()}")

    train_df = train_df[train_df["label"] >= 0].copy()
    train_df["label"] = train_df["label"].astype(int)

    print("Label distribution:")
    print(train_df["label"].value_counts().rename({0:"model_a", 1:"model_b", 2:"tie"}))

    print("Building train features...")
    train_feat = build_feature_df(train_df)

    print("Training XGBoost model...")
    model, feature_cols = train_xgb(train_feat, train_df["label"])
else:
    print("No train.csv found — using heuristic scoring only.")


Loading train data...
  Train samples: 57477
Columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
Label distribution:
label
model_a    20064
model_b    19652
tie        17761
Name: count, dtype: int64
Building train features...
Training XGBoost model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:02:04] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost trained on 57477 samples with 48 features.


In [8]:
print("Generating predictions...")
predictions = []

if model is not None:
    X_test = test_feat[feature_cols].fillna(0)
    proba  = model.predict_proba(X_test)  # shape (n, 3): [a, b, tie]
    for i, (_, row) in enumerate(test_df.iterrows()):
        predictions.append({
            "id":             row["id"],
            "winner_model_a": proba[i, 0],
            "winner_model_b": proba[i, 1],
            "winner_tie":     proba[i, 2],
        })
else:
    for _, row in test_feat.iterrows():
        sa, sb = heuristic_score(row)
        pa, pb, pt = scores_to_probs(sa, sb, tie_weight=0.22)
        predictions.append({
            "id":             row["id"],
            "winner_model_a": pa,
            "winner_model_b": pb,
            "winner_tie":     pt,
        })

sub = pd.DataFrame(predictions)

# Sanity check
row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-5), "Probabilities do not sum to 1!"

print("\nSample predictions:")
print(sub.head(10).to_string(index=False))

Generating predictions...

Sample predictions:
     id  winner_model_a  winner_model_b  winner_tie
 136060        0.295444        0.284539    0.420017
 211333        0.604917        0.185875    0.209208
1233961        0.175509        0.541204    0.283287


In [9]:
out_path = OUTPUT_DIR / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"\n✓ Saved to {out_path}")
sub.head()


✓ Saved to /kaggle/working/submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.295444,0.284539,0.420017
1,211333,0.604917,0.185875,0.209208
2,1233961,0.175509,0.541204,0.283287
